# Cauldron Dataset VLM Evaluation

This notebook evaluates VLM models on samples from the [HuggingFaceM4/the_cauldron](https://huggingface.co/datasets/HuggingFaceM4/the_cauldron) dataset.

**Pipeline:**
1. Load dataset samples (streaming mode)
2. Extract image + prompt + ground truth
3. Run inference across all configured VLM models
4. Compare predictions to ground truth
5. Compute metrics (accuracy, latency, cost)

## 1. Setup & Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
from pathlib import Path
import uuid
import re
from datetime import datetime
from typing import List, Dict, Any, Optional

import pandas as pd
from datasets import load_dataset, get_dataset_config_names
from IPython.display import display, Image as IPImage
from tqdm.auto import tqdm

# Add parent to path for local imports
sys.path.append(os.path.abspath(".."))

In [ ]:
from inference_api_call.client import WhichVLMClient

## 2. Configuration

In [ ]:
# === PATHS ===
CONFIG_PATH = (Path.cwd().parent / "configs" / "inference_vlm.yaml").resolve()
DATA_ROOT = Path("../../../dataset/which_vlm_data").resolve()
IMAGES_DIR = DATA_ROOT / "images" / "cauldron"
OUTPUT_DIR = DATA_ROOT / "results"

# Create directories
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Config path: {CONFIG_PATH}")
print(f"Data root: {DATA_ROOT}")
print(f"Images dir: {IMAGES_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# === EVALUATION CONFIG ===
CAULDRON_REPO = "HuggingFaceM4/the_cauldron"

# Which subsets to evaluate
TARGET_CONFIGS = ["textvqa", "ai2d", "chartqa", "vqav2"]

# Samples per config (keep small for testing)
SAMPLES_PER_CONFIG = 10

# Generation parameters
GEN_KWARGS = {
    "temperature": 0.1,
    "max_tokens": 256,
}

## 3. Initialize VLM Client

In [ ]:
client = WhichVLMClient.from_yaml(CONFIG_PATH)

print("All models:", client.list_models())
print("VLM models:", client.list_vlm_models())
print("LLM models:", client.list_llm_models())

## 4. Dataset Loading Utilities

In [ ]:
def return_dataset_configs(repo: str) -> List[str]:
    """
    Return available dataset config names for a given dataset repo.
    """
    return get_dataset_config_names(repo)

def load_cauldron_samples(config_name: str, n_samples: int = 32) -> List[Dict]:
    """
    Load samples from a Cauldron subset using streaming mode.
    
    Returns a list of dicts with keys: 'images', 'texts'
    """
    ds = load_dataset(
        CAULDRON_REPO,
        config_name,
        streaming=True,
        # trust_remote_code=True
    )
    print(f"Number of configs available: {get_dataset_config_names(CAULDRON_REPO)}")
    samples = list(ds['train'].take(n_samples))
    print(f"Loaded {len(samples)} samples from '{config_name}'")
    return samples


def extract_qa_from_sample(sample: Dict) -> Dict[str, Any]:
    """
    Extract image, user prompt, and ground truth from a Cauldron sample.
    
    Cauldron format:
        sample['images'] = [PIL.Image, ...]
        sample['texts'] = [{'user': '...', 'assistant': '...', 'source': '...'}, ...]
    
    Returns:
        {
            'image': PIL.Image,
            'prompt': str,
            'ground_truth': str,
            'source': str
        }
    """
    if not sample.get('images') or not sample.get('texts'):
        return None
    
    image = sample['images'][0]
    text_turn = sample['texts'][0]
    
    return {
        'image': image,
        'prompt': text_turn['user'],
        'ground_truth': text_turn['assistant'],
        'source': text_turn.get('source', 'unknown')
    }


def extract_answer_letter(text: str) -> Optional[str]:
    """
    Extract answer letter (A, B, C, D) from multiple choice responses.
    Handles formats like 'Answer: A', '(A)', 'A.', 'A)', etc.
    """
    patterns = [
        r"Answer:\s*([A-D])",
        r"\(([A-D])\)",
        r"^([A-D])[\.\)]",
        r"^([A-D])$",
    ]
    for pattern in patterns:
        match = re.search(pattern, text.strip(), re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None

In [ ]:
# Quick test: load a few samples
test_samples = load_cauldron_samples("textvqa", n_samples=2)

for i, sample in enumerate(test_samples):
    qa = extract_qa_from_sample(sample)
    if qa:
        print(f"\n--- Sample {i} ---")
        print(f"Prompt: {qa['prompt'][:100]}...")
        print(f"Ground Truth: {qa['ground_truth']}")
        display(qa['image'].resize((200, 200)))

## 5. Single Sample Inference

In [ ]:
def run_inference_on_sample(
    client: WhichVLMClient,
    qa: Dict[str, Any],
    models: str = "all",
    **gen_kwargs
) -> Dict[str, Dict[str, Any]]:
    """
    Run VLM inference on a single QA sample.
    
    Args:
        client: WhichVLMClient instance
        qa: Dict with 'image', 'prompt', 'ground_truth'
        models: "all" or list of model names
        **gen_kwargs: Generation parameters
    
    Returns:
        Dict mapping model_name -> result dict with added 'ground_truth' key
    """
    results = client.vlm.run_image(
        image=qa['image'],
        text=qa['prompt'],
        models=models,
        **gen_kwargs
    )
    
    # Add ground truth to each result for easy comparison
    for model_name in results:
        results[model_name]['ground_truth'] = qa['ground_truth']
        results[model_name]['prompt'] = qa['prompt']
    
    return results

In [ ]:
# Test single inference
test_qa = extract_qa_from_sample(test_samples[1])

print("Prompt:", test_qa['prompt'])
print("Ground Truth:", test_qa['ground_truth'])
display(test_qa['image'].resize((300, 300)))

# Run inference
results = run_inference_on_sample(client, test_qa, **GEN_KWARGS)

print("\n" + "="*50)
for model_name, out in results.items():
    print(f"\n=== {model_name} ===")
    if out['ok']:
        print(f"Response: {out['response_text']}")
        print(f"Latency: {out['latency_ms']}ms")
    else:
        print(f"ERROR: {out.get('error')}")

## 6. Batch Evaluation Pipeline

In [ ]:
def evaluate_config(
    client: WhichVLMClient,
    config_name: str,
    n_samples: int = 32,
    save_images: bool = True,
    **gen_kwargs
) -> pd.DataFrame:
    """
    Evaluate all VLM models on samples from a single Cauldron config.
    
    Returns a DataFrame with one row per (sample, model) pair.
    """
    samples = load_cauldron_samples(config_name, n_samples)
    vlm_models = client.list_vlm_models()
    
    rows = []
    
    for sample_idx, sample in enumerate(tqdm(samples, desc=f"Evaluating {config_name}")):
        qa = extract_qa_from_sample(sample)
        if qa is None:
            continue
        
        # Optionally save image to disk
        image_path = None
        if save_images:
            uid = str(uuid.uuid4())
            image_path = IMAGES_DIR / config_name / f"{uid}.png"
            image_path.parent.mkdir(parents=True, exist_ok=True)
            qa['image'].save(image_path)
            image_path = str(image_path)
        
        # Run inference on all VLM models
        try:
            results = run_inference_on_sample(client, qa, models="all", **gen_kwargs)
        except Exception as e:
            print(f"Error on sample {sample_idx}: {e}")
            continue
        
        # Create one row per model
        for model_name, out in results.items():
            row = {
                'sample_id': sample_idx,
                'config': config_name,
                'model': model_name,
                'prompt': qa['prompt'],
                'ground_truth': qa['ground_truth'],
                'image_path': image_path,
                'ok': out.get('ok', False),
                'response_text': out.get('response_text', ''),
                'latency_ms': out.get('latency_ms', 0),
                'est_cost': out.get('est_cost', 0.0),
                'error': out.get('error', None),
            }
            rows.append(row)
    
    df = pd.DataFrame(rows)
    return df

In [ ]:
# Evaluate on a single config first
df_textvqa = evaluate_config(
    client,
    config_name="textvqa",
    n_samples=SAMPLES_PER_CONFIG,
    **GEN_KWARGS
)

print(f"\nResults shape: {df_textvqa.shape}")
df_textvqa.head(10)

## 7. Scoring & Metrics

In [ ]:
def normalize_answer(text: str) -> str:
    """Normalize text for comparison: lowercase, strip, remove punctuation."""
    if not text:
        return ""
    text = text.lower().strip()
    # Remove trailing punctuation
    text = re.sub(r'[.!?,;:]+$', '', text)
    return text


def exact_match(pred: str, gt: str) -> bool:
    """Check if prediction exactly matches ground truth (normalized)."""
    return normalize_answer(pred) == normalize_answer(gt)


def contains_match(pred: str, gt: str) -> bool:
    """Check if ground truth is contained in prediction (normalized)."""
    pred_norm = normalize_answer(pred)
    gt_norm = normalize_answer(gt)
    return gt_norm in pred_norm


def add_scores(df: pd.DataFrame) -> pd.DataFrame:
    """Add scoring columns to results DataFrame."""
    df = df.copy()
    
    df['exact_match'] = df.apply(
        lambda r: exact_match(r['response_text'], r['ground_truth']) if r['ok'] else False,
        axis=1
    )
    
    df['contains_match'] = df.apply(
        lambda r: contains_match(r['response_text'], r['ground_truth']) if r['ok'] else False,
        axis=1
    )
    
    # For multiple choice: extract letter answers
    df['pred_letter'] = df['response_text'].apply(extract_answer_letter)
    df['gt_letter'] = df['ground_truth'].apply(extract_answer_letter)
    df['letter_match'] = (df['pred_letter'] == df['gt_letter']) & df['pred_letter'].notna()
    
    return df

In [ ]:
# Add scores to our results
df_scored = add_scores(df_textvqa)

# Show sample comparisons
print("Sample predictions vs ground truth:")
display(df_scored[['model', 'ground_truth', 'response_text', 'exact_match', 'contains_match']].head(10))

In [ ]:
def compute_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute aggregate metrics per model.
    
    Returns a DataFrame with one row per model.
    """
    # Filter to successful responses only
    df_ok = df[df['ok'] == True]
    
    metrics = df_ok.groupby('model').agg(
        n_samples=('sample_id', 'count'),
        exact_match_acc=('exact_match', 'mean'),
        contains_match_acc=('contains_match', 'mean'),
        letter_match_acc=('letter_match', 'mean'),
        avg_latency_ms=('latency_ms', 'mean'),
        p50_latency_ms=('latency_ms', 'median'),
        p95_latency_ms=('latency_ms', lambda x: x.quantile(0.95)),
        total_cost=('est_cost', 'sum'),
    ).round(4)
    
    # Add error rate
    error_rates = df.groupby('model')['ok'].apply(lambda x: 1 - x.mean()).rename('error_rate')
    metrics = metrics.join(error_rates)
    
    return metrics.reset_index()


metrics_textvqa = compute_metrics(df_scored)
print(f"\n=== Metrics for textvqa ({SAMPLES_PER_CONFIG} samples) ===")
display(metrics_textvqa)

## 8. Multi-Config Evaluation

In [ ]:
def evaluate_multiple_configs(
    client: WhichVLMClient,
    configs: List[str],
    n_samples: int = 32,
    **gen_kwargs
) -> pd.DataFrame:
    """
    Evaluate all VLM models across multiple Cauldron configs.
    """
    all_dfs = []
    
    for config_name in configs:
        print(f"\n{'='*50}")
        print(f"Evaluating: {config_name}")
        print('='*50)
        
        try:
            df = evaluate_config(
                client,
                config_name=config_name,
                n_samples=n_samples,
                **gen_kwargs
            )
            all_dfs.append(df)
            print(f" {config_name}: {len(df)} results")
        except Exception as e:
            print(f" {config_name}: ERROR - {e}")
    
    if not all_dfs:
        return pd.DataFrame()
    
    df_all = pd.concat(all_dfs, ignore_index=True)
    return df_all

In [ ]:
# Run evaluation across all target configs
# ️ This may take a while depending on your models and sample count!

df_all = evaluate_multiple_configs(
    client,
    configs=TARGET_CONFIGS,
    n_samples=SAMPLES_PER_CONFIG,
    **GEN_KWARGS
)

print(f"\nTotal results: {len(df_all)} rows")
print(f"Configs: {df_all['config'].unique().tolist()}")
print(f"Models: {df_all['model'].unique().tolist()}")

In [ ]:
# Add scores and compute metrics
df_all_scored = add_scores(df_all)

# Overall metrics per model
print("=== Overall Metrics (All Configs) ===")
metrics_overall = compute_metrics(df_all_scored)
display(metrics_overall)

In [ ]:
# Metrics broken down by config
print("\n=== Metrics by Config ===")

for config in df_all_scored['config'].unique():
    df_cfg = df_all_scored[df_all_scored['config'] == config]
    metrics = compute_metrics(df_cfg)
    print(f"\n--- {config} ---")
    display(metrics[['model', 'n_samples', 'exact_match_acc', 'contains_match_acc', 'avg_latency_ms']])

## 9. Save Results

In [ ]:
# Save detailed results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

results_path = OUTPUT_DIR / f"cauldron_eval_{timestamp}.parquet"
df_all_scored.to_parquet(results_path, index=False)
print(f"Saved results to: {results_path}")

# Save metrics summary
metrics_path = OUTPUT_DIR / f"cauldron_metrics_{timestamp}.csv"
metrics_overall.to_csv(metrics_path, index=False)
print(f"Saved metrics to: {metrics_path}")

## 10. Inspect Individual Results

In [ ]:
def show_sample_comparison(df: pd.DataFrame, sample_id: int):
    """
    Display a single sample with all model predictions side by side.
    """
    sample_df = df[df['sample_id'] == sample_id]
    if sample_df.empty:
        print(f"No sample found with id={sample_id}")
        return
    
    row = sample_df.iloc[0]
    
    # Show image if available
    if row['image_path'] and Path(row['image_path']).exists():
        display(IPImage(filename=row['image_path'], width=400))
    
    print(f"Config: {row['config']}")
    print(f"Prompt: {row['prompt']}")
    print(f"Ground Truth: {row['ground_truth']}")
    print("\n" + "-"*50)
    
    for _, r in sample_df.iterrows():
        status = "" if r['exact_match'] else ("~" if r['contains_match'] else "")
        print(f"\n{status} {r['model']}:")
        print(f"   {r['response_text'][:200]}{'...' if len(str(r['response_text'])) > 200 else ''}")
        print(f"   (latency: {r['latency_ms']}ms)")

In [ ]:
# Show a few sample comparisons
for sid in [0, 1, 2]:
    print("\n" + "="*60)
    show_sample_comparison(df_all_scored, sid)

In [ ]:
# Find samples where models disagree
def find_disagreements(df: pd.DataFrame) -> List[int]:
    """Find sample_ids where models have different exact_match results."""
    disagreements = []
    
    for sid in df['sample_id'].unique():
        sample_df = df[df['sample_id'] == sid]
        matches = sample_df['exact_match'].unique()
        if len(matches) > 1:  # Some models got it right, others didn't
            disagreements.append(sid)
    
    return disagreements

disagreement_ids = find_disagreements(df_all_scored)
print(f"Found {len(disagreement_ids)} samples with model disagreements")

# Show first disagreement
if disagreement_ids:
    print("\n" + "="*60)
    print("Example disagreement:")
    show_sample_comparison(df_all_scored, disagreement_ids[0])

## 11. Summary

In [ ]:
print("="*60)
print("EVALUATION SUMMARY")
print("="*60)
print(f"\nDataset: {CAULDRON_REPO}")
print(f"Configs evaluated: {TARGET_CONFIGS}")
print(f"Samples per config: {SAMPLES_PER_CONFIG}")
print(f"Total samples: {df_all_scored['sample_id'].nunique()}")
print(f"Models evaluated: {df_all_scored['model'].unique().tolist()}")

print("\n" + "-"*60)
print("MODEL RANKINGS (by contains_match accuracy):")
print("-"*60)

rankings = metrics_overall.sort_values('contains_match_acc', ascending=False)
for i, (_, row) in enumerate(rankings.iterrows(), 1):
    print(f"{i}. {row['model']}: {row['contains_match_acc']*100:.1f}% accuracy, {row['avg_latency_ms']:.0f}ms avg latency")